In [1]:
#commentaire

#commentaite2

In [2]:
#Ajout des modifs avec cherry-pick

In [1]:
!pip install -U datasets

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
from transformers import AutoImageProcessor, ConvNextV2ForImageClassification
import torch
from datasets import load_dataset

dataset = load_dataset("Alwaly/Oral_Cancer-cancer")
image = dataset["train"]["image"][0]

preprocessor = AutoImageProcessor.from_pretrained("facebook/convnextv2-base-22k-224")
model = ConvNextV2ForImageClassification.from_pretrained("facebook/convnextv2-base-22k-224")

inputs = preprocessor(image, return_tensors="pt")

with torch.no_grad():
    logits = model(**inputs).logits

# model predicts one of the 1000 ImageNet classes
predicted_label = logits.argmax(-1).item()
print(model.config.id2label[predicted_label]),


In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 10002
    })
})

FINETUNING

In [6]:
from tqdm.notebook import tqdm
import torch
from transformers import AutoModelForImageClassification, AutoImageProcessor
from torchvision.transforms import (
    Compose,
    Normalize,
    RandomHorizontalFlip,
    RandomResizedCrop,
    ToTensor,
)
from datasets import load_dataset, ClassLabel
from torch.utils.data import DataLoader

In [7]:
dataset = load_dataset("Alwaly/Oral_Cancer-cancer")

#Encodage
id2label = {0: "oral_normal", 1: "oral_scc"}
label2id = {"oral_normal": 0, "oral_scc": 1}
labels = ["oral_normal", "oral_scc"]

In [8]:
image_processor = AutoImageProcessor.from_pretrained("facebook/convnextv2-base-22k-224")

In [9]:
image_processor

ConvNextImageProcessor {
  "crop_pct": 0.875,
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.485,
    0.456,
    0.406
  ],
  "image_processor_type": "ConvNextImageProcessor",
  "image_std": [
    0.229,
    0.224,
    0.225
  ],
  "resample": 3,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "shortest_edge": 224
  }
}

In [10]:
normaliser = Normalize(mean=image_processor.image_mean, std=image_processor.image_std)

In [11]:
transform = Compose([RandomResizedCrop(image_processor.size['shortest_edge']), RandomHorizontalFlip(), ToTensor(), normaliser])

In [12]:
def train_transform(example_batch):
    example_batch["pixel_values"] = [transform(image.convert("RGB")) for image in example_batch["image"]]
    return example_batch

In [13]:
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 10002
    })
})

In [14]:
dataset = dataset['train'].train_test_split(test_size=0.2)

In [15]:
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 8001
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 2001
    })
})

In [16]:
dataset['validation'] = dataset['test'].train_test_split(test_size=0.5)['train']

In [17]:
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 8001
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 2001
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 1000
    })
})

In [18]:
dataset['test'] = dataset['test'].train_test_split(test_size=0.5)['test']

In [19]:
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 8001
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 1001
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 1000
    })
})

In [20]:
processed_dataset = dataset.with_transform(train_transform)

In [21]:
def collate_fn(examples):
  pixel_values = torch.stack([example['pixel_values']for example in examples])
  labels = torch.tensor([label2id[example['label']]for example in examples])
  return {"pixel_values": pixel_values, "labels": labels}

In [22]:
data_loader = DataLoader(processed_dataset['train'], batch_size=16, shuffle=True, collate_fn=collate_fn)

In [23]:
model = AutoModelForImageClassification.from_pretrained(
    "facebook/convnextv2-base-22k-224",
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

Some weights of ConvNextV2ForImageClassification were not initialized from the model checkpoint at facebook/convnextv2-base-22k-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 1024]) in the checkpoint and torch.Size([2, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [24]:
optimiser = torch.optim.AdamW(model.parameters(), lr=5e-5)

In [25]:
torch.cuda.is_available()

True

In [26]:
# if torch.cuda.is_available():
#     device = 'cuda'
# else : device = 'cpu'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [27]:
model.to(device)

ConvNextV2ForImageClassification(
  (convnextv2): ConvNextV2Model(
    (embeddings): ConvNextV2Embeddings(
      (patch_embeddings): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
      (layernorm): ConvNextV2LayerNorm()
    )
    (encoder): ConvNextV2Encoder(
      (stages): ModuleList(
        (0): ConvNextV2Stage(
          (downsampling_layer): Identity()
          (layers): Sequential(
            (0): ConvNextV2Layer(
              (dwconv): Conv2d(128, 128, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=128)
              (layernorm): ConvNextV2LayerNorm()
              (pwconv1): Linear(in_features=128, out_features=512, bias=True)
              (act): GELUActivation()
              (grn): ConvNextV2GRN()
              (pwconv2): Linear(in_features=512, out_features=128, bias=True)
              (drop_path): Identity()
            )
            (1): ConvNextV2Layer(
              (dwconv): Conv2d(128, 128, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=

In [ ]:
model.train()
epochs = 1
for epoch in range(epochs):
    correct = 0
    total = 0
    for idx, batch in enumerate(tqdm(data_loader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimiser.zero_grad()
        #Forward Propagation
        outputs = model(pixel_values=batch['pixel_values'], labels=batch['labels'])
        loss = outputs.loss
        logits = outputs.logits
        #Back propagation
        loss.backward()
        #Update
        optimiser.step()
        #Metrics
        total = total+batch['labels'].shape[0]
        predicted = logits.argmax(-1)
        correct = correct+(predicted==batch['labels']).sum().item()
        accuracy = correct/total
        if idx%20 == 0:
          print(f'Loss apres {idx} steps : {loss.item()}')
          print(f'Accuracy apres {idx} steps : {accuracy}')


  0%|          | 0/501 [00:00<?, ?it/s]

Loss apres 0 steps : 0.7504321932792664
Accuracy apres 0 steps : 0.4375
Loss apres 20 steps : 0.6014676094055176
Accuracy apres 20 steps : 0.6369047619047619
Loss apres 40 steps : 0.42032718658447266
Accuracy apres 40 steps : 0.7134146341463414


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
repo_name = 'kassemabo77/oral_cancer_detection'
image_processor.push_to_hub(repo_name)
model.push_to_hub(repo_name)